# Kaggle Stage-1 Exact Replication Runner (Single Notebook)

This notebook is designed for a **fresh Kaggle notebook** with only your dataset attached.

It performs end-to-end Stage-1 setup and run:
1. Clone repos.
2. Normalize paths to the author-expected layout.
3. Copy + canonicalize `.dat` files.
4. Recreate `test_set.pkl` from provided train/test loader pickles.
5. Apply a compatibility shim for public `imagen-pytorch` API differences.
6. Run Stage-1 training + evaluation with live progress and optional W&B.

Model architecture and training logic are not changed; only runtime compatibility and observability are added.


## Step 0: Fill Exactly One Placeholder

Edit only this line in the next cell:
- `DATASET_ROOT = Path("/kaggle/input/<YOUR_DATASET_SLUG_HERE>")`

Expected dataset structure:
- `/kaggle/input/<slug>/dataloaders_fixed/*.dat`
- `/kaggle/input/<slug>/split_dataloaders/train_loader.pkl`
- `/kaggle/input/<slug>/split_dataloaders/test_loader.pkl`


In [21]:
from pathlib import Path

# ========================
# REQUIRED USER INPUT
# ========================
DATASET_ROOT = Path('/kaggle/input/final-dataset-with-splits')

# ========================
# RUN CONFIG
# ========================
RUN_NAME = 'v_FC_dim64'
EPOCHS = 100

REPO_URL = 'https://github.com/vedanggggg/cyclone-forecasting'
REPO_DIR = Path('/kaggle/working/forecast-diffmodels')

IMAGEN_REPO_URL = 'https://github.com/lucidrains/imagen-pytorch.git'
IMAGEN_PYTORCH_DIR = Path('/kaggle/working/imagen-pytorch')

ENABLE_WANDB = True
WANDB_PROJECT = "cyclone-forecasting save and run notebook"
WANDB_ENTITY = None   # or your exact wandb username slug
from kaggle_secrets import UserSecretsClient
WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
WANDB_MODE = "online" if WANDB_API_KEY else "offline"


DATA_DATALOADERS = DATASET_ROOT / 'dataloaders_fixed'
DATA_TRAIN_PKL = DATASET_ROOT / 'split_dataloaders' / 'train_loader.pkl'
DATA_TEST_PKL = DATASET_ROOT / 'split_dataloaders' / 'test_loader.pkl'

required = [DATASET_ROOT, DATA_DATALOADERS, DATA_TRAIN_PKL, DATA_TEST_PKL]
missing = [p for p in required if not p.exists()]
assert not missing, 'Missing required paths:\n' + '\n'.join(str(x) for x in missing)

print('Input validation passed.')
print('Dataset root      :', DATASET_ROOT)
print('Dat files found   :', len(list(DATA_DATALOADERS.glob('*.dat'))))
print('Train split pickle:', DATA_TRAIN_PKL)
print('Test split pickle :', DATA_TEST_PKL)


Input validation passed.
Dataset root      : /kaggle/input/final-dataset-with-splits
Dat files found   : 15
Train split pickle: /kaggle/input/final-dataset-with-splits/split_dataloaders/train_loader.pkl
Test split pickle : /kaggle/input/final-dataset-with-splits/split_dataloaders/test_loader.pkl


## Step 1: Clone Repositories


In [6]:
import subprocess

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already exists:', REPO_DIR)

if not IMAGEN_PYTORCH_DIR.exists():
    subprocess.run(['git', 'clone', IMAGEN_REPO_URL, str(IMAGEN_PYTORCH_DIR)], check=True)
else:
    print('imagen-pytorch already exists:', IMAGEN_PYTORCH_DIR)


Repo already exists: /kaggle/working/forecast-diffmodels
imagen-pytorch already exists: /kaggle/working/imagen-pytorch


## Step 2: Detect Real Project Root and Normalize Folder Names


In [7]:
import os
from pathlib import Path

# Fix accidental trailing-space folder names like "imagen "
for p in REPO_DIR.rglob('imagen '):
    target = p.parent / 'imagen'
    if not target.exists():
        os.rename(p, target)
        print('Renamed:', p, '->', target)

candidate_roots = []
for root in [REPO_DIR, *(p for p in REPO_DIR.iterdir() if p.is_dir())]:
    if (root / 'dataproc' / 'utils.py').exists() and (root / 'imagen' / 'helpers.py').exists() and (root / 'imagen' / '64_FC' / 'train64.py').exists():
        candidate_roots.append(root)

for utils_path in REPO_DIR.rglob('dataproc/utils.py'):
    root = utils_path.parent.parent
    if (root / 'imagen' / 'helpers.py').exists() and (root / 'imagen' / '64_FC' / 'train64.py').exists():
        candidate_roots.append(root)

candidate_roots = sorted(set(candidate_roots), key=lambda x: len(str(x)))
assert candidate_roots, 'Could not find project root containing dataproc/utils.py + imagen/helpers.py + imagen/64_FC/train64.py'

PROJECT_ROOT = candidate_roots[0]
DATAPROC_DIR = PROJECT_ROOT / 'dataproc'
IMAGEN_DIR = PROJECT_ROOT / 'imagen'
STAGE1_DIR = IMAGEN_DIR / '64_FC'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATAPROC_DIR =', DATAPROC_DIR)
print('IMAGEN_DIR   =', IMAGEN_DIR)
print('STAGE1_DIR   =', STAGE1_DIR)


PROJECT_ROOT = /kaggle/working/forecast-diffmodels/forecast-video-diffmodels
DATAPROC_DIR = /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/dataproc
IMAGEN_DIR   = /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/imagen
STAGE1_DIR   = /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/imagen/64_FC


## Step 3: Install Dependencies

This installs only what is needed for Stage-1 run + monitoring.


In [22]:
import sys, subprocess, shutil
from pathlib import Path

IMAGEN_PYTORCH_DIR = Path('/kaggle/working/imagen-pytorch')

# 1) Re-clone cleanly
if IMAGEN_PYTORCH_DIR.exists():
    shutil.rmtree(IMAGEN_PYTORCH_DIR)
subprocess.run(
    ['git', 'clone', 'https://github.com/lucidrains/imagen-pytorch.git', str(IMAGEN_PYTORCH_DIR)],
    check=True
)

# 2) Install with full logs (no -q)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(IMAGEN_PYTORCH_DIR)], check=True)

# 3) Install remaining deps
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'einops', 'pixelmatch', 'torchmetrics', 'lpips',
    'xarray', 'satpy', 'fsspec', 'pyproj', 'scipy', 'scikit-image',
    'openpyxl', 'dill', 'pandas', 'tensorboard', 'opencv-python', 'wandb'
], check=True)

# 4) Import check (robust for Kaggle editable installs)
import sys
if str(IMAGEN_PYTORCH_DIR) not in sys.path:
    sys.path.insert(0, str(IMAGEN_PYTORCH_DIR))

import imagen_pytorch
print('imagen_pytorch import OK:', imagen_pytorch.__file__)


import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "wandb>=0.22.3"], check=True)

import wandb
print("wandb version:", wandb.__version__)



Cloning into '/kaggle/working/imagen-pytorch'...


Obtaining file:///kaggle/working/imagen-pytorch
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for imagen-pytorch (pyproject.toml): started
  Building editable for imagen-pytorch (pyproject.toml): finished with status 'done'
  Created wheel for imagen-pytorch: filename=imagen_pytorch-2.1.0-0.editable-py3-none-any.whl size=4413 sha256=b1a4a22fe4f8edc42bcbfa07e2d0d3c2e2773d2f18e82ee1ecba780a37f92ee4
  Stored in directory: /tmp/pip-ephem-wheel-cache-zv03poqh/wheels/64/4f/49/1568cfb45cb33b40c658af28881eff3a191ebdf39fd47bae1b
Succ

## Step 4: Recreate Author-Expected `/rds/...` Paths


In [13]:
import os
import shutil

RDS_HOME = Path('/rds/general/user/zr523/home/researchProject')
RDS_DATA = Path('/rds/general/ephemeral/user/zr523/ephemeral')
RDS_PROJECT_LINK = RDS_HOME / 'forecast-diffmodels'
RDS_DATALOADER_64_FC = RDS_DATA / 'satellite' / 'dataloader' / '64_FC'

RDS_HOME.mkdir(parents=True, exist_ok=True)
RDS_DATALOADER_64_FC.mkdir(parents=True, exist_ok=True)

if RDS_PROJECT_LINK.is_symlink() or RDS_PROJECT_LINK.exists():
    if RDS_PROJECT_LINK.is_symlink() or RDS_PROJECT_LINK.is_file():
        RDS_PROJECT_LINK.unlink()
    else:
        shutil.rmtree(RDS_PROJECT_LINK)
os.symlink(str(PROJECT_ROOT), str(RDS_PROJECT_LINK))

print('Symlink:', RDS_PROJECT_LINK, '->', os.readlink(RDS_PROJECT_LINK))
print('Dataloader dir:', RDS_DATALOADER_64_FC)


Symlink: /rds/general/user/zr523/home/researchProject/forecast-diffmodels -> /kaggle/working/forecast-diffmodels/forecast-video-diffmodels
Dataloader dir: /rds/general/ephemeral/user/zr523/ephemeral/satellite/dataloader/64_FC


## Step 5: Copy and Canonicalize Dat Files

Input naming currently: `cyclone_region.dat`.

Author code expects: `region_cyclone.dat`.


In [14]:
import shutil

for old in RDS_DATALOADER_64_FC.glob('*.dat'):
    old.unlink()

copied = []
for src in sorted(DATA_DATALOADERS.glob('*.dat')):
    stem = src.stem
    assert '_' in stem, f'Unexpected filename format: {src.name}'
    cyclone, region = stem.rsplit('_', 1)
    dst = RDS_DATALOADER_64_FC / f'{region}_{cyclone}.dat'
    shutil.copy2(src, dst)
    copied.append(dst.name)

print('Copied dat files:', len(copied))
print('Sample:', copied[:6])


Copied dat files: 15
Sample: ['use_bonnie.dat', 'use_delta.dat', 'use_eta.dat', 'use_fiona.dat', 'usw_genevieve.dat', 'use_grace.dat']


## Step 6: Reconstruct `test_set.pkl` from Provided Split Pickles

This avoids assumptions and recreates the same train/test sample counts as your provided split files.


In [15]:
import pickle
from itertools import combinations

# Minimal placeholder classes for loading pickles saved from notebook context
class CycloneDataLoader: pass
class ModelDataLoader: pass

with open(DATA_TEST_PKL, 'rb') as f:
    test_loader_ref = pickle.load(f)
with open(DATA_TRAIN_PKL, 'rb') as f:
    train_loader_ref = pickle.load(f)

target_test = int(test_loader_ref.img_o.shape[0])
target_train = int(train_loader_ref.img_o.shape[0])

items = []
for fp in sorted(RDS_DATALOADER_64_FC.glob('*.dat')):
    region, name = fp.stem.split('_', 1)
    with open(fp, 'rb') as f:
        obj = pickle.load(f)
    count = int(obj.img_64.shape[0])
    items.append((region, name, count, fp))

total = sum(x[2] for x in items)
assert total == (target_train + target_test), f'Total mismatch: dat_total={total}, expected={target_train + target_test}'

idxs = list(range(len(items)))
matches = []
for r in range(1, len(items) + 1):
    for comb in combinations(idxs, r):
        test_count = sum(items[i][2] for i in comb)
        train_count = total - test_count
        if test_count == target_test and train_count == target_train:
            matches.append(comb)

assert matches, 'No cyclone subset matched provided train/test sample counts.'
matches = sorted(matches, key=lambda x: (len(x), [items[i][1] for i in x]))
chosen = matches[0]

split = {'nio': [], 'aus': [], 'wpo': [], 'wio': [], 'use': [], 'usw': []}
for i in chosen:
    region, name, _, _ = items[i]
    split[region].append(name)

if len(matches) > 1:
    print('WARNING: multiple valid test subsets found. Using smallest + lexical order by default.')
    print('Num valid subsets:', len(matches))

test_set_local = DATAPROC_DIR / 'test_set.pkl'
with open(test_set_local, 'wb') as f:
    pickle.dump(split, f)

test_set_rds = RDS_PROJECT_LINK / 'dataproc' / 'test_set.pkl'
with open(test_set_rds, 'wb') as f:
    pickle.dump(split, f)

print('test_set.pkl written:', test_set_local)
print('test split mapping :', split)
print('target train/test  :', target_train, target_test)


Num valid subsets: 18
test_set.pkl written: /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/dataproc/test_set.pkl
test split mapping : {'nio': [], 'aus': [], 'wpo': [], 'wio': [], 'use': ['delta', 'eta', 'laura'], 'usw': []}
target train/test  : 2229 816


## Step 7: Apply Runtime Compatibility Patch (No Architecture Change)

Public `imagen-pytorch` removed `condition_on_continuous` and `continuous_embeds` APIs.
This shim maps those to current text-conditioning internals so original script logic can run unchanged.


In [16]:
from pathlib import Path

compat_path = IMAGEN_DIR / "imagen_api_compat.py"

compat_code = r'''
import torch
from imagen_pytorch import Unet, Unet3D, Imagen as _Imagen, ImagenTrainer as _ImagenTrainer, NullUnet


def _to_text_condition(continuous_embeds):
    if continuous_embeds is None:
        return None, None

    if not torch.is_tensor(continuous_embeds):
        continuous_embeds = torch.tensor(continuous_embeds)

    if continuous_embeds.ndim == 1:
        continuous_embeds = continuous_embeds.unsqueeze(0)

    if continuous_embeds.ndim > 2:
        continuous_embeds = continuous_embeds.reshape(continuous_embeds.shape[0], -1)

    # one token per sample -> [batch, 1, dim]
    text_embeds = continuous_embeds.float().unsqueeze(1)
    text_masks = torch.ones(
        (text_embeds.shape[0], text_embeds.shape[1]),
        dtype=torch.bool,
        device=text_embeds.device
    )
    return text_embeds, text_masks


class Imagen(_Imagen):
    def __init__(self, *args, condition_on_continuous=False, continuous_embed_dim=None, **kwargs):
        if condition_on_continuous:
            kwargs.setdefault("condition_on_text", True)
            if continuous_embed_dim is not None:
                kwargs.setdefault("text_embed_dim", continuous_embed_dim)
        super().__init__(*args, **kwargs)

    def sample(self, *args, continuous_embeds=None, **kwargs):
        text_embeds, text_masks = _to_text_condition(continuous_embeds)
        if text_embeds is not None and "text_embeds" not in kwargs:
            kwargs["text_embeds"] = text_embeds
        if text_masks is not None and "text_masks" not in kwargs:
            kwargs["text_masks"] = text_masks
        return super().sample(*args, **kwargs)


class ImagenTrainer(_ImagenTrainer):
    def __call__(self, *args, continuous_embeds=None, **kwargs):
        text_embeds, text_masks = _to_text_condition(continuous_embeds)
        if text_embeds is not None and "text_embeds" not in kwargs:
            kwargs["text_embeds"] = text_embeds
        if text_masks is not None and "text_masks" not in kwargs:
            kwargs["text_masks"] = text_masks
        return super().__call__(*args, **kwargs)
'''.lstrip()

compat_path.write_text(compat_code)
print("Wrote:", compat_path)

def ensure_import_patch(path: Path, old: str, new: str):
    text = path.read_text()
    if new in text:
        return
    if old in text:
        text = text.replace(old, new)
        path.write_text(text)

utils_py = DATAPROC_DIR / "utils.py"
train_py = STAGE1_DIR / "train64.py"
eval_py = STAGE1_DIR / "v_t02-sampling-and-evaluation.py"
test_py = STAGE1_DIR / "test64.py"

old_utils = "from imagen_pytorch import Unet, Unet3D, Imagen, ImagenTrainer, NullUnet"
new_utils = (
    "try:\n"
    "    from imagen_api_compat import Unet, Unet3D, Imagen, ImagenTrainer, NullUnet\n"
    "except ImportError:\n"
    "    from imagen_pytorch import Unet, Unet3D, Imagen, ImagenTrainer, NullUnet"
)

old_train = "from imagen_pytorch import Unet3D, Imagen, ImagenTrainer"
new_train = (
    "try:\n"
    "    from imagen_api_compat import Unet3D, Imagen, ImagenTrainer\n"
    "except ImportError:\n"
    "    from imagen_pytorch import Unet3D, Imagen, ImagenTrainer"
)

old_eval_test = "from imagen_pytorch import Unet3D, Imagen, ImagenTrainer, NullUnet"
new_eval_test = (
    "try:\n"
    "    from imagen_api_compat import Unet3D, Imagen, ImagenTrainer, NullUnet\n"
    "except ImportError:\n"
    "    from imagen_pytorch import Unet3D, Imagen, ImagenTrainer, NullUnet"
)

ensure_import_patch(utils_py, old_utils, new_utils)
ensure_import_patch(train_py, old_train, new_train)
ensure_import_patch(eval_py, old_eval_test, new_eval_test)
ensure_import_patch(test_py, old_eval_test, new_eval_test)

print("Import patching done.")


Wrote: /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/imagen/imagen_api_compat.py
Patched files count: 1
- /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/dataproc/utils.py


## Step 8: Validate Imports and Dataloader Construction


In [19]:
import sys

for p in [DATAPROC_DIR, IMAGEN_DIR, IMAGEN_PYTORCH_DIR]:
    ps = str(p)
    if ps not in sys.path:
        sys.path.insert(0, ps)

import utils
from helpers import get_satellite_data

class Args: pass
args = Args()
args.batch_size = 1
args.o_size = 64
args.n_size = 128
args.dataset_path = str(RDS_DATALOADER_64_FC)
args.datalimit = False
args.mode = 'fc'
args.lr = 3e-4
args.augment = False

train_dl, test_dl = get_satellite_data(args, 'vid')
print('Import + dataloader build OK')

# v_ModelDataLoader uses img / vid fields (not img_o)
print('Train img shape :', train_dl.img.shape)
print('Test img shape  :', test_dl.img.shape)
print('Train vid shape :', train_dl.vid.shape)
print('Test vid shape  :', test_dl.vid.shape)

print('Train img count :', train_dl.img.shape[0])
print('Test img count  :', test_dl.img.shape[0])

assert train_dl.img.shape[0] == 2229, f"Unexpected train count: {train_dl.img.shape[0]}"
assert test_dl.img.shape[0] == 816, f"Unexpected test count: {test_dl.img.shape[0]}"
print('Count check passed: train=2229, test=816')


 13%|█▎        | 2/15 [00:00<00:00, 19.70it/s]

laura
fiona
ida


 27%|██▋       | 4/15 [00:00<00:00, 11.11it/s]

ian
sally


 40%|████      | 6/15 [00:00<00:01,  5.62it/s]

...

bonnie


 93%|█████████▎| 14/15 [00:05<00:00,  1.39it/s]

olaf


100%|██████████| 15/15 [00:06<00:00,  2.41it/s]

Import + dataloader build OK
Train img shape : torch.Size([2229, 64, 64])
Test img shape  : torch.Size([816, 64, 64])
Train vid shape : torch.Size([216, 10, 64, 64])
Test vid shape  : torch.Size([81, 10, 64, 64])
Train img count : 2229
Test img count  : 816


## Step 9: Train Stage-1 with Live Progress and Optional W&B

This cell streams subprocess output and prints periodic monitor status:
- elapsed time
- latest log line
- checkpoint count
- GPU utilization


In [24]:
from pathlib import Path

compat_path = Path("/kaggle/working/forecast-diffmodels/forecast-video-diffmodels/imagen/imagen_api_compat.py")

compat_code = r'''
import torch
from imagen_pytorch import Unet, Unet3D, Imagen as _Imagen, ImagenTrainer as _ImagenTrainer, NullUnet


def _to_text_condition(continuous_embeds):
    if continuous_embeds is None:
        return None, None

    if not torch.is_tensor(continuous_embeds):
        continuous_embeds = torch.tensor(continuous_embeds)

    if continuous_embeds.ndim == 1:
        continuous_embeds = continuous_embeds.unsqueeze(0)

    if continuous_embeds.ndim > 2:
        continuous_embeds = continuous_embeds.reshape(continuous_embeds.shape[0], -1)

    # one token per sample -> [batch, 1, dim]
    text_embeds = continuous_embeds.float().unsqueeze(1)
    text_masks = torch.ones(
        (text_embeds.shape[0], text_embeds.shape[1]),
        dtype=torch.bool,
        device=text_embeds.device
    )
    return text_embeds, text_masks


class Imagen(_Imagen):
    def __init__(self, *args, condition_on_continuous=False, continuous_embed_dim=None, **kwargs):
        if condition_on_continuous:
            kwargs.setdefault("condition_on_text", True)
            if continuous_embed_dim is not None:
                kwargs.setdefault("text_embed_dim", continuous_embed_dim)
        super().__init__(*args, **kwargs)

    def sample(self, *args, continuous_embeds=None, **kwargs):
        text_embeds, text_masks = _to_text_condition(continuous_embeds)
        if text_embeds is not None and "text_embeds" not in kwargs:
            kwargs["text_embeds"] = text_embeds
        if text_masks is not None and "text_masks" not in kwargs:
            kwargs["text_masks"] = text_masks
        return super().sample(*args, **kwargs)


class ImagenTrainer(_ImagenTrainer):
    def __call__(self, *args, continuous_embeds=None, **kwargs):
        text_embeds, text_masks = _to_text_condition(continuous_embeds)
        if text_embeds is not None and "text_embeds" not in kwargs:
            kwargs["text_embeds"] = text_embeds
        if text_masks is not None and "text_masks" not in kwargs:
            kwargs["text_masks"] = text_masks
        return super().__call__(*args, **kwargs)
'''.lstrip()

compat_path.write_text(compat_code)
print("Wrote:", compat_path)
print("First line:", compat_path.read_text().splitlines()[0])


Wrote: /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/imagen/imagen_api_compat.py
First line: import torch


In [ ]:
import os
import re
import glob
import time
import queue
import threading
import subprocess
from datetime import datetime
import matplotlib.pyplot as plt


def supports_flag(script_path: Path, flag: str) -> bool:
    return flag in script_path.read_text()


def gpu_status():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total', '--format=csv,noheader,nounits'],
            text=True
        ).strip().splitlines()
        if not out:
            return 'GPU info unavailable'
        util, mem_used, mem_total = [x.strip() for x in out[0].split(',')]
        return f'gpu={util}% mem={mem_used}/{mem_total} MiB'
    except Exception:
        return 'GPU info unavailable'


def run_with_monitor(cmd, cwd: Path, env: dict, run_name: str, report_every_sec: int = 30):
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    q = queue.Queue()

    def _reader(pipe, qobj):
        for line in iter(pipe.readline, ''):
            qobj.put(line.rstrip('\n'))
        pipe.close()

    t = threading.Thread(target=_reader, args=(proc.stdout, q), daemon=True)
    t.start()

    start = time.time()
    last_report = 0.0
    history_t = []
    history_ckpt = []

    run_log = RDS_HOME / 'models' / run_name / 'run.log'
    ckpt_glob = str(RDS_HOME / 'models' / run_name / 'models' / run_name / 'ckpt_1_*.pt')

    print('[monitor] command:', ' '.join(cmd))
    print('[monitor] cwd    :', cwd)

    while True:
        drained = False
        while True:
            try:
                line = q.get_nowait()
                drained = True
                if line.strip():
                    print(line)
            except queue.Empty:
                break

        now = time.time()
        if now - last_report >= report_every_sec:
            ckpts = sorted(glob.glob(ckpt_glob))
            latest_ckpt = Path(ckpts[-1]).name if ckpts else 'none'
            latest_log = ''
            if run_log.exists():
                try:
                    latest_log = run_log.read_text().strip().splitlines()[-1]
                except Exception:
                    latest_log = ''

            elapsed_min = (now - start) / 60.0
            history_t.append(elapsed_min)
            history_ckpt.append(len(ckpts))

            print(f"[monitor] t={elapsed_min:.1f}m ckpts={len(ckpts)} latest={latest_ckpt} {gpu_status()}")
            if latest_log:
                print(f"[monitor] log: {latest_log}")

            last_report = now

        rc = proc.poll()
        if rc is not None:
            while True:
                try:
                    line = q.get_nowait()
                    if line.strip():
                        print(line)
                except queue.Empty:
                    break

            if rc != 0:
                raise subprocess.CalledProcessError(rc, cmd)

            if history_t:
                plt.figure(figsize=(7, 3))
                plt.plot(history_t, history_ckpt, marker='o')
                plt.title('Checkpoint Count Over Time')
                plt.xlabel('Elapsed minutes')
                plt.ylabel('Saved checkpoint files')
                plt.grid(True, alpha=0.3)
                plt.show()
            print('[monitor] process completed successfully.')
            break

        time.sleep(1)


env = os.environ.copy()
env['PYTHONPATH'] = f"{DATAPROC_DIR}:{IMAGEN_DIR}:{IMAGEN_PYTORCH_DIR}:{env.get('PYTHONPATH','')}"
env['PYTHONUNBUFFERED'] = '1'
env['FDM_PROJECT_ROOT'] = str(PROJECT_ROOT)
env['FDM_BASE_HOME'] = str(RDS_PROJECT_LINK)
env['FDM_BASE_DATA'] = str(RDS_DATA)
env['FDM_DATAPROC_DIR'] = str(DATAPROC_DIR)
env['FDM_IMAGEN_DIR'] = str(IMAGEN_DIR)
env['FDM_TEST_SET_PATH'] = str(DATAPROC_DIR / 'test_set.pkl')

if ENABLE_WANDB and WANDB_API_KEY:
    env['WANDB_API_KEY'] = WANDB_API_KEY

train_script = STAGE1_DIR / 'train64.py'
train_cmd = ['python', '-u', 'train64.py', '-mode', 'execute', '-run_name', RUN_NAME, '-epochs', str(EPOCHS)]

if '--progress_interval' in train_script.read_text():
    train_cmd += ['--progress_interval', '25']

if ENABLE_WANDB and supports_flag(train_script, '--enable_wandb'):
    train_cmd += ['--enable_wandb', '--wandb_project', WANDB_PROJECT, '--wandb_mode', WANDB_MODE, '--wandb_run_name', WANDB_RUN_NAME_TRAIN]
    if WANDB_ENTITY:
        train_cmd += ['--wandb_entity', WANDB_ENTITY]

run_with_monitor(train_cmd, STAGE1_DIR, env, RUN_NAME, report_every_sec=30)


[monitor] command: python -u train64.py -mode execute -run_name v_FC_dim64 -epochs 100 --progress_interval 25 --enable_wandb --wandb_project cyclone-forecasting --wandb_mode online --wandb_run_name v_FC_dim64-train
[monitor] cwd    : /kaggle/working/forecast-diffmodels/forecast-video-diffmodels/imagen/64_FC
[monitor] t=0.0m ckpts=0 latest=none gpu=0% mem=3/15360 MiB
2026-02-08 10:49:17.921333: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770547757.945498     451 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770547757.952494     451 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770547757.971197     451 computation_placer.cc:177] com

## Step 10: Run Evaluation with Live Progress and Optional W&B


In [ ]:
import os
import subprocess


env = os.environ.copy()
env['PYTHONPATH'] = f"{DATAPROC_DIR}:{IMAGEN_DIR}:{IMAGEN_PYTORCH_DIR}:{env.get('PYTHONPATH','')}"
env['PYTHONUNBUFFERED'] = '1'
env['FDM_PROJECT_ROOT'] = str(PROJECT_ROOT)
env['FDM_BASE_HOME'] = str(RDS_PROJECT_LINK)
env['FDM_BASE_DATA'] = str(RDS_DATA)
env['FDM_DATAPROC_DIR'] = str(DATAPROC_DIR)
env['FDM_IMAGEN_DIR'] = str(IMAGEN_DIR)
env['FDM_TEST_SET_PATH'] = str(DATAPROC_DIR / 'test_set.pkl')

if ENABLE_WANDB and WANDB_API_KEY:
    env['WANDB_API_KEY'] = WANDB_API_KEY

script = STAGE1_DIR / 'v_t02-sampling-and-evaluation.py'
cmd = ['python', '-u', 'v_t02-sampling-and-evaluation.py', '-run_name', RUN_NAME]
text = script.read_text()

if '--progress_interval' in text:
    cmd += ['--progress_interval', '1']
if ENABLE_WANDB and '--enable_wandb' in text:
    cmd += ['--enable_wandb', '--wandb_project', WANDB_PROJECT, '--wandb_mode', WANDB_MODE, '--wandb_run_name', WANDB_RUN_NAME_EVAL]
    if WANDB_ENTITY:
        cmd += ['--wandb_entity', WANDB_ENTITY]

run_with_monitor(cmd, STAGE1_DIR, env, RUN_NAME, report_every_sec=30)


## Step 11 (Optional): Test-Set Metrics at Latest Checkpoint


In [ ]:
import glob
import re
import os
import subprocess

ckpt_dir = RDS_HOME / 'models' / RUN_NAME / 'models' / RUN_NAME
ckpt_files = sorted(glob.glob(str(ckpt_dir / 'ckpt_1_*.pt')))
if not ckpt_files:
    print('No checkpoints found; skipping test64.')
else:
    latest_epoch = max(int(re.search(r'_(\d{3})\.pt$', x).group(1)) for x in ckpt_files)
    print('Using BEST_EPOCH =', latest_epoch)

    env = os.environ.copy()
    env['PYTHONPATH'] = f"{DATAPROC_DIR}:{IMAGEN_DIR}:{IMAGEN_PYTORCH_DIR}:{env.get('PYTHONPATH','')}"
    env['PYTHONUNBUFFERED'] = '1'
    env['FDM_PROJECT_ROOT'] = str(PROJECT_ROOT)
    env['FDM_BASE_HOME'] = str(RDS_PROJECT_LINK)
    env['FDM_BASE_DATA'] = str(RDS_DATA)
    env['FDM_DATAPROC_DIR'] = str(DATAPROC_DIR)
    env['FDM_IMAGEN_DIR'] = str(IMAGEN_DIR)
    env['FDM_TEST_SET_PATH'] = str(DATAPROC_DIR / 'test_set.pkl')

    cmd = ['python', '-u', 'test64.py', '-run_name', RUN_NAME, '-best_epoch', str(latest_epoch)]
    run_with_monitor(cmd, STAGE1_DIR, env, RUN_NAME, report_every_sec=30)


## Step 12: Output Locations


In [ ]:
from pathlib import Path

model_root = Path('/rds/general/user/zr523/home/researchProject/models') / RUN_NAME
print('Run log          :', model_root / 'run.log')
print('Model checkpoints:', model_root / 'models' / RUN_NAME)
print('Result images    :', model_root / 'results' / RUN_NAME)
print('Eval metrics pkl :', model_root / 'metrics.pkl')
print('Test metrics pkl :', model_root / 'metrics_test.pkl')
print('test_set.pkl     :', DATAPROC_DIR / 'test_set.pkl')
